In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VALIDASI WAVEFORM VENEZUELA
Memeriksa:
1. Integritas file (dapat dibaca oleh ObsPy)
2. Kecocokan antara file dan katalog (berdasarkan waktu)
3. Ketersediaan komponen (Z, N, E)
4. Posisi origin time dalam rentang waveform
5. File yang hilang (berdasarkan katalog)
"""

import os
import sys
import pandas as pd
from pathlib import Path
from obspy import read, UTCDateTime
from tqdm import tqdm
import logging

# =============================================
# KONFIGURASI
# =============================================
WAVEFORM_DIR = '/Volumes/Extreme SSD/venezuela_data_earthquake/waveform_venezuela_3c'
CATALOG_CSV = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/venezuela_earthquake_bench/query_venezuela.csv"
OUTPUT_REPORT = "/Volumes/Extreme SSD/venezuela_data_earthquake/validation_report.csv"
OUTPUT_SUMMARY = "/Volumes/Extreme SSD/venezuela_data_earthquake/validation_summary.txt"

# =============================================
# SETUP LOGGING
# =============================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# =============================================
# FUNGSI VALIDASI
# =============================================

def parse_event_id_from_filename(filename):
    """
    Ekstrak event_id (waktu) dari nama file.
    Format: {NET}_{STA}_{YYYYMMDD}_{HHMMSS}.mseed
    Contoh: CU_GRGR_20260628_084610.mseed -> "20260628_084610"
    """
    stem = filename.stem  # tanpa ekstensi
    parts = stem.split('_')
    if len(parts) >= 4:
        # Ambil 2 bagian terakhir (YYYYMMDD dan HHMMSS)
        return '_'.join(parts[-2:])
    return None

def read_catalog(catalog_path):
    """Baca katalog dan buat dictionary untuk akses cepat."""
    df = pd.read_csv(catalog_path)
    # Konversi kolom time ke datetime
    df['origin_dt'] = pd.to_datetime(df['time'], utc=True)
    # Buat key string YYYYMMDD_HHMMSS untuk pencocokan
    df['time_str'] = df['origin_dt'].dt.strftime("%Y%m%d_%H%M%S")
    # Buat dictionary
    catalog_dict = df.set_index('time_str').to_dict('index')
    logger.info(f"📋 Katalog: {len(df)} event, rentang {df['origin_dt'].min()} - {df['origin_dt'].max()}")
    return df, catalog_dict

def validate_file(file_path, catalog_dict):
    """
    Validasi satu file .mseed.
    Return dict dengan status dan metadata.
    """
    result = {
        'filename': file_path.name,
        'event_id': None,
        'network': None,
        'station': None,
        'in_catalog': False,
        'readable': False,
        'has_Z': False,
        'has_N': False,
        'has_E': False,
        'trace_count': 0,
        'sampling_rate': None,
        'starttime': None,
        'endtime': None,
        'origin_time': None,
        'origin_inside': False,
        'error_message': None,
        'status': 'UNKNOWN'
    }
    
    # Ekstrak event_id dari nama file
    event_id = parse_event_id_from_filename(file_path)
    if event_id is None:
        result['status'] = 'INVALID_FILENAME'
        result['error_message'] = 'Format nama file tidak sesuai'
        return result
    result['event_id'] = event_id
    
    # Cek di katalog
    if event_id in catalog_dict:
        result['in_catalog'] = True
        result['origin_time'] = catalog_dict[event_id]['origin_dt']
    else:
        result['status'] = 'NOT_IN_CATALOG'
        return result
    
    # Baca header waveform (headonly=True sangat cepat)
    try:
        st = read(str(file_path), headonly=True)
        result['readable'] = True
        result['trace_count'] = len(st)
        
        if len(st) == 0:
            result['status'] = 'EMPTY_FILE'
            return result
        
        # Ambil trace pertama untuk info dasar
        tr = st[0]
        result['network'] = tr.stats.network
        result['station'] = tr.stats.station
        result['sampling_rate'] = tr.stats.sampling_rate
        result['starttime'] = tr.stats.starttime
        result['endtime'] = tr.stats.endtime
        
        # Periksa komponen
        channels = [t.stats.channel for t in st]
        result['has_Z'] = any(ch.endswith('Z') for ch in channels)
        result['has_N'] = any(ch.endswith('N') for ch in channels)
        result['has_E'] = any(ch.endswith('E') for ch in channels)
        
        # Periksa apakah origin time berada di dalam rekaman
        origin = result['origin_time']
        if result['starttime'] <= origin <= result['endtime']:
            result['origin_inside'] = True
            result['status'] = 'VALID'
        else:
            result['status'] = 'ORIGIN_OUTSIDE'
            result['error_message'] = f"Origin {origin} di luar rentang [{result['starttime']} - {result['endtime']}]"
            
    except Exception as e:
        result['readable'] = False
        result['status'] = 'CORRUPTED'
        result['error_message'] = str(e)
    
    return result

def main():
    logger.info("="*60)
    logger.info("🔍 VALIDASI WAVEFORM VENEZUELA")
    logger.info("="*60)
    
    # 1. Baca katalog
    catalog_df, catalog_dict = read_catalog(CATALOG_CSV)
    
    # 2. Cari semua file .mseed
    wave_dir = Path(WAVEFORM_DIR)
    all_files = list(wave_dir.glob("*.mseed"))
    logger.info(f"📁 Ditemukan {len(all_files)} file .mseed")
    
    # 3. Validasi setiap file
    results = []
    for file_path in tqdm(all_files, desc="Validasi file"):
        result = validate_file(file_path, catalog_dict)
        results.append(result)
    
    # 4. Buat DataFrame
    df_results = pd.DataFrame(results)
    
    # 5. Identifikasi file yang hilang (ada di katalog tapi tidak ada file)
    catalog_keys = set(catalog_dict.keys())
    file_keys = set(df_results[df_results['in_catalog'] == True]['event_id'].dropna())
    missing_files = catalog_keys - file_keys
    
    logger.info(f"\n📊 RINGKASAN VALIDASI:")
    logger.info(f"   Total file di direktori: {len(all_files)}")
    logger.info(f"   File yang valid (lolos semua): {len(df_results[df_results['status'] == 'VALID'])}")
    logger.info(f"   File korup/tidak terbaca: {len(df_results[df_results['status'] == 'CORRUPTED'])}")
    logger.info(f"   File tidak ada di katalog: {len(df_results[df_results['status'] == 'NOT_IN_CATALOG'])}")
    logger.info(f"   Origin time di luar rekaman: {len(df_results[df_results['status'] == 'ORIGIN_OUTSIDE'])}")
    logger.info(f"   File hilang (ada di katalog tapi tidak diunduh): {len(missing_files)}")
    
    # 6. Detail komponen
    valid_files = df_results[df_results['status'] == 'VALID']
    if len(valid_files) > 0:
        logger.info(f"\n📦 KOMPOSISI KOMPONEN (dari file valid):")
        has_3comp = valid_files[(valid_files['has_Z']) & (valid_files['has_N']) & (valid_files['has_E'])]
        logger.info(f"   Memiliki 3 komponen (Z, N, E): {len(has_3comp)}")
        logger.info(f"   Hanya Z: {len(valid_files[valid_files['has_Z'] & ~valid_files['has_N'] & ~valid_files['has_E']])}")
        logger.info(f"   Hanya Z + N: {len(valid_files[valid_files['has_Z'] & valid_files['has_N'] & ~valid_files['has_E']])}")
        logger.info(f"   Hanya Z + E: {len(valid_files[valid_files['has_Z'] & ~valid_files['has_N'] & valid_files['has_E']])}")
    
    # 7. Simpan laporan detail ke CSV
    df_results.to_csv(OUTPUT_REPORT, index=False)
    logger.info(f"\n💾 Laporan detail disimpan di: {OUTPUT_REPORT}")
    
    # 8. Simpan ringkasan ke file teks
    with open(OUTPUT_SUMMARY, 'w') as f:
        f.write("="*60 + "\n")
        f.write("RINGKASAN VALIDASI WAVEFORM VENEZUELA\n")
        f.write("="*60 + "\n\n")
        f.write(f"Total file di direktori: {len(all_files)}\n")
        f.write(f"File valid: {len(df_results[df_results['status'] == 'VALID'])}\n")
        f.write(f"File korup: {len(df_results[df_results['status'] == 'CORRUPTED'])}\n")
        f.write(f"File tidak ada di katalog: {len(df_results[df_results['status'] == 'NOT_IN_CATALOG'])}\n")
        f.write(f"Origin di luar rekaman: {len(df_results[df_results['status'] == 'ORIGIN_OUTSIDE'])}\n")
        f.write(f"File hilang dari katalog: {len(missing_files)}\n\n")
        
        if missing_files:
            f.write("DAFTAR EVENT YANG TIDAK MEMILIKI FILE WAVEFORM (HILANG):\n")
            for mf in sorted(missing_files):
                f.write(f"  - {mf}\n")
        
        f.write("\n" + "="*60 + "\n")
    
    logger.info(f"💾 Ringkasan disimpan di: {OUTPUT_SUMMARY}")
    
    # 9. Tampilkan contoh event hilang (jika ada)
    if missing_files:
        logger.info(f"\n⚠️  Contoh 10 event yang tidak memiliki file:")
        for mf in sorted(missing_files)[:10]:
            logger.info(f"   - {mf}")
    
    logger.info("="*60)
    logger.info("✅ VALIDASI SELESAI")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-07-15 10:32:02,911 - INFO - ============================================================
2026-07-15 10:32:02,911 - INFO - 🔍 VALIDASI WAVEFORM VENEZUELA
2026-07-15 10:32:02,912 - INFO - ============================================================
2026-07-15 10:32:02,933 - INFO - 📋 Katalog: 745 event, rentang 2001-08-11 15:23:07.330000+00:00 - 2026-07-07 08:41:37.390000+00:00
2026-07-15 10:32:02,942 - INFO - 📁 Ditemukan 1790 file .mseed


Validasi file: 100%|██████████| 1790/1790 [00:07<00:00, 234.63it/s]

2026-07-15 10:32:10,602 - INFO - 
📊 RINGKASAN VALIDASI:
2026-07-15 10:32:10,602 - INFO -    Total file di direktori: 1790
2026-07-15 10:32:10,603 - INFO -    File yang valid (lolos semua): 895
2026-07-15 10:32:10,604 - INFO -    File korup/tidak terbaca: 895
2026-07-15 10:32:10,604 - INFO -    File tidak ada di katalog: 0
2026-07-15 10:32:10,605 - INFO -    Origin time di luar rekaman: 0
2026-07-15 10:32:10,605 - INFO -    File hilang (ada di katalog tapi tidak diunduh): 484
2026-07-15 10:32:10,606 - INFO - 
📦 KOMPOSISI KOMPONEN (dari file valid):
2026-07-15 10:32:10,606 - INFO -    Memiliki 3 komponen (Z, N, E): 895
2026-07-15 10:32:10,606 - INFO -    Hanya Z: 0
2026-07-15 10:32:10,607 - INFO -    Hanya Z + N: 0
2026-07-15 10:32:10,607 - INFO -    Hanya Z + E: 0
2026-07-15 10:32:10,624 - INFO - 
💾 Laporan detail disimpan di: /Volumes/Extreme SSD/venezuela_data_earthquake/validation_report.csv
2026-07-15 10:32:10,626 - INFO - 💾 Ringkasan disimpan di: /Volumes/Extreme SSD/venezuela_data